# Ingest Constructors JSON File

This notebook demonstrates the ingestion of the constructors.json file into the bronze layer.

## What this notebook covers
* Load the constructors.json source file
* Add ingestion metadata columns: source_file and ingestion_timestamp
* Write the transformed data to the bronze Delta table
* Validate the loaded results

This uses centralized configuration and helper functions for consistency.

In [0]:
%run ../00-common/01.Formula1_Configuration_Setup

In [0]:
# Check configured catalog
catalog_name

In [0]:
%run ../00-common/02.bronze-helper

In [0]:
# Check landing folder path
landing_folder_path

In [0]:
# Define source file and target table
source_file = f"{landing_folder_path}/constructors.json"
table_name = f"{catalog_name}.{bronze_schema}.constructors"

In [0]:
# Preview source file path
source_file

In [0]:
# Define the schema
constructors_schema = """constructorId STRING,name STRING,nationality STRING,url STRING"""

In [0]:
# Read constructors JSON into a DataFrame
constructors_df = (
    spark.read
    .format("json")
    .option("multiLine", True)
    .option("mode", "FAILFAST")
    .load(source_file)
)

display(constructors_df)

In [0]:
# Add ingestion metadata columns
constructors_final_df = add_ingestion_metadata(constructors_df)
display(constructors_final_df)

In [0]:
# Write constructors data to the bronze Delta table
(
    constructors_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

In [0]:
# Preview target table name
table_name

In [0]:
%sql
-- Query the bronze constructors table
SELECT * FROM formula1.bronze.constructors

In [0]:
# Display the bronze constructors table
display(spark.table(table_name))